# Comparación: Tatoeba vs. OPUS-100 (dataset chico vs. grande)

**Curso:** Tópicos Avanzados en Machine Learning — Proyecto de Neural Networks

Este notebook compara los dos modelos entrenados con la misma arquitectura Transformer (desde cero) y los
mismos hiperparámetros, cambiando únicamente el dataset de entrenamiento:

- **Tatoeba** (`mt_en_es_transformer.ipynb`) — dataset chico, oraciones cortas y coloquiales.
- **OPUS-100** (`mt_en_es_transformer_opus100.ipynb`) — dataset mucho más grande, dominio mixto.

**Antes de ejecutar**, sube a esta sesión de Colab los archivos que generó cada notebook:

De Tatoeba: `history_tatoeba.json`, `resultados_test.csv`, `transformer_en_es.pth`, `vocab_en.json`,
`vocab_es.json`, `model_config.json` (los 4 últimos son opcionales — solo si quieres la sección 4, que
traduce las mismas oraciones con ambos modelos).

De OPUS-100: `history_opus100.json`, `resultados_test_opus100.csv`, `transformer_en_es_opus100.pth`,
`vocab_en_opus100.json`, `vocab_es_opus100.json`, `model_config_opus100.json` (mismo criterio).

## 1. Cargar resultados de ambas corridas

In [ ]:
import json, math
import pandas as pd
import matplotlib.pyplot as plt

with open('history_tatoeba.json', encoding='utf-8') as f:
    hist_a = json.load(f)
with open('history_opus100.json', encoding='utf-8') as f:
    hist_b = json.load(f)

df_a = pd.read_csv('resultados_test.csv')
df_b = pd.read_csv('resultados_test_opus100.csv')

print('Tatoeba  -> test BLEU (recalculado):', df_a['bleu'].mean())
print('OPUS-100 -> test BLEU (recalculado):', df_b['bleu'].mean())

## 2. Tabla comparativa

In [ ]:
comparison = pd.DataFrame([
    {
        'Dataset': hist_a['dataset'],
        'Pares de train': hist_a['n_train'],
        'Pares de test': hist_a['n_test'],
        'Vocab. inglés': hist_a['vocab_en'],
        'Vocab. español': hist_a['vocab_es'],
        'Parámetros': hist_a['n_params'],
        'Épocas': len(hist_a['train_loss']),
        's/época (prom.)': sum(hist_a['epoch_seconds']) / len(hist_a['epoch_seconds']),
        'Val_loss final': hist_a['best_val_loss'],
        'BLEU en test': df_a['bleu'].mean(),
    },
    {
        'Dataset': hist_b['dataset'],
        'Pares de train': hist_b['n_train'],
        'Pares de test': hist_b['n_test'],
        'Vocab. inglés': hist_b['vocab_en'],
        'Vocab. español': hist_b['vocab_es'],
        'Parámetros': hist_b['n_params'],
        'Épocas': len(hist_b['train_loss']),
        's/época (prom.)': sum(hist_b['epoch_seconds']) / len(hist_b['epoch_seconds']),
        'Val_loss final': hist_b['best_val_loss'],
        'BLEU en test': df_b['bleu'].mean(),
    },
])
comparison.to_csv('comparacion_datasets.csv', index=False)
comparison

## 3. Curvas de pérdida de validación, lado a lado

In [ ]:
plt.figure(figsize=(7,4.5))
plt.plot(range(1, len(hist_a['val_loss'])+1), hist_a['val_loss'], marker='o', label=f"{hist_a['dataset']} (val)")
plt.plot(range(1, len(hist_b['val_loss'])+1), hist_b['val_loss'], marker='o', label=f"{hist_b['dataset']} (val)")
plt.xlabel('Época'); plt.ylabel('Val loss (CrossEntropy)')
plt.title('Curva de validación: dataset chico vs. grande')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig('comparacion_loss.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Traducir las mismas oraciones con ambos modelos (opcional)

Requiere los 4 archivos de cada modelo (`.pth` + 2 vocabularios + config). Si no los subiste, salta esta
sección — la tabla y el gráfico de arriba ya son suficientes para la comparación cuantitativa del reporte.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        Q = self.w_q(q).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.w_k(k).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.w_v(v).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, -1, self.n_heads * self.d_k)
        return self.w_o(out)


class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model))

    def forward(self, x):
        return self.net(x)

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        x = self.norm2(x + self.dropout(self.ff(x)))
        return x


class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, dropout=0.1, max_len=100):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model

    def forward(self, src, mask):
        x = self.embed(src) * math.sqrt(self.d_model)
        x = self.dropout(self.pos_enc(x))
        for layer in self.layers:
            x = layer(x, mask)
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask, tgt_mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.dropout(self.cross_attn(x, enc_out, enc_out, src_mask)))
        x = self.norm3(x + self.dropout(self.ff(x)))
        return x


class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, dropout=0.1, max_len=100):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model

    def forward(self, tgt, enc_out, src_mask, tgt_mask):
        x = self.embed(tgt) * math.sqrt(self.d_model)
        x = self.dropout(self.pos_enc(x))
        for layer in self.layers:
            x = layer(x, enc_out, src_mask, tgt_mask)
        return self.fc_out(x)


class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=256, n_heads=8, d_ff=512,
                 n_layers=3, dropout=0.1, max_len=100, pad_idx=0):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, d_model, n_heads, d_ff, n_layers, dropout, max_len)
        self.decoder = Decoder(tgt_vocab_size, d_model, n_heads, d_ff, n_layers, dropout, max_len)
        self.pad_idx = pad_idx

    def make_src_mask(self, src):
        return (src != self.pad_idx).unsqueeze(1).unsqueeze(2)

    def make_tgt_mask(self, tgt):
        pad_mask = (tgt != self.pad_idx).unsqueeze(1).unsqueeze(2)
        L = tgt.size(1)
        sub_mask = torch.tril(torch.ones((L, L), device=tgt.device)).bool()
        return pad_mask & sub_mask

    def forward(self, src, tgt):
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)
        enc_out = self.encoder(src, src_mask)
        return self.decoder(tgt, enc_out, src_mask, tgt_mask)

In [ ]:
import re, torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN = '<pad>', '<sos>', '<eos>', '<unk>'

def normalize_text(s):
    s = s.strip().lower()
    s = re.sub(r"([.!?¿¡,])", r" \1 ", s)
    s = re.sub(r"[^a-zñáéíóúü¿¡.!?, ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

class Vocab:
    def __init__(self, word2idx):
        self.word2idx = word2idx
        self.idx2word = {i: w for w, i in word2idx.items()}

    def encode(self, sentence, add_sos_eos=True):
        ids = [self.word2idx.get(w, self.word2idx[UNK_TOKEN]) for w in sentence.split()]
        if add_sos_eos:
            ids = [self.word2idx[SOS_TOKEN]] + ids + [self.word2idx[EOS_TOKEN]]
        return ids

    def decode(self, ids):
        words = []
        for i in ids:
            w = self.idx2word.get(int(i), UNK_TOKEN)
            if w == EOS_TOKEN:
                break
            if w in (SOS_TOKEN, PAD_TOKEN):
                continue
            words.append(w)
        return ' '.join(words)

    def __len__(self):
        return len(self.word2idx)

def load_model(pth_path, vocab_en_path, vocab_es_path, config_path):
    with open(vocab_en_path, encoding='utf-8') as f:
        src_vocab = Vocab(json.load(f))
    with open(vocab_es_path, encoding='utf-8') as f:
        tgt_vocab = Vocab(json.load(f))
    with open(config_path) as f:
        config = json.load(f)
    pad_idx = src_vocab.word2idx[PAD_TOKEN]
    model = Transformer(len(src_vocab), len(tgt_vocab), pad_idx=pad_idx, **config).to(device)
    model.load_state_dict(torch.load(pth_path, map_location=device))
    model.eval()
    return model, src_vocab, tgt_vocab

def translate(model, src_vocab, tgt_vocab, sentence, max_len=30):
    norm = normalize_text(sentence)
    src_ids = torch.tensor([src_vocab.encode(norm)], dtype=torch.long).to(device)
    src_mask = model.make_src_mask(src_ids)
    with torch.no_grad():
        enc_out = model.encoder(src_ids, src_mask)
    tgt_ids = [tgt_vocab.word2idx[SOS_TOKEN]]
    for _ in range(max_len):
        tgt_tensor = torch.tensor([tgt_ids], dtype=torch.long).to(device)
        tgt_mask = model.make_tgt_mask(tgt_tensor)
        with torch.no_grad():
            out = model.decoder(tgt_tensor, enc_out, src_mask, tgt_mask)
        next_id = out[0, -1].argmax().item()
        tgt_ids.append(next_id)
        if next_id == tgt_vocab.word2idx[EOS_TOKEN]:
            break
    return tgt_vocab.decode(tgt_ids[1:])

model_a, src_vocab_a, tgt_vocab_a = load_model('transformer_en_es.pth', 'vocab_en.json', 'vocab_es.json', 'model_config.json')
model_b, src_vocab_b, tgt_vocab_b = load_model('transformer_en_es_opus100.pth', 'vocab_en_opus100.json', 'vocab_es_opus100.json', 'model_config_opus100.json')
print('Ambos modelos cargados.')

In [ ]:
ejemplos = [
    'I love you.',
    'What time is it?',
    'The weather is nice today.',
    'Where is the nearest hospital?',
    'I am learning machine learning.',
    'A drunk driver was responsible for the car accident.',
]

rows = []
for s in ejemplos:
    rows.append({
        'Oración (inglés)': s,
        f"{hist_a['dataset']}": translate(model_a, src_vocab_a, tgt_vocab_a, s),
        f"{hist_b['dataset']}": translate(model_b, src_vocab_b, tgt_vocab_b, s),
    })

pd.set_option('display.max_colwidth', None)
pd.DataFrame(rows)

## 5. Conclusiones de la comparación

*(Completar con los números reales de la sección 2 y las observaciones de la sección 4 antes de
entregar el reporte. Preguntas guía:)*

- ¿El BLEU en test mejoró con más datos? ¿En qué proporción, respecto a cuánto creció el dataset?
- ¿La curva de validación del modelo grande sigue bajando al final del entrenamiento (señal de que le
  faltarían más épocas) o ya se aplanó como la del modelo chico?
- En los ejemplos de la sección 4: ¿el modelo de OPUS-100 maneja mejor vocabulario fuera del dominio de
  Tatoeba (ej. *machine learning*, *drunk driver*)? ¿Sigue apareciendo el loop de repetición?
- ¿Vale la pena, en términos de tiempo de entrenamiento vs. mejora de BLEU, usar el dataset más grande?